# DVC Quickstart

Welcome! In this notebook, you'll get hands-on experience with DVC, a tool for versioning datasets and models.

**Objective:** version a dataset and a model with DVC, go back to a previous version, and (optionally) push the data to a remote storage, in about 15 minutes.

## 1. Why DVC?

Git is designed for source code: small text files whose changes can be read line by line. Data and models are different:

- **They are large.** GitHub refuses files over 100 MB:
  ```
  remote: error: File large_dataset.csv is 159.77 MB; this exceeds GitHub's file size limit of 100.00 MB
  ```
- **They are often binary**, so a line-by-line diff is meaningless.
- **Every version stored in Git stays in its history forever**, making the repository heavier at each change.

DVC (Data Version Control) splits the problem in two:

- the **data** is stored in the DVC cache (and later in a remote storage: S3, Google Drive, DagsHub...);
- Git only versions a small **`.dvc` metafile** containing the hash of the data.

This way, each Git commit points to an exact version of the data.

## 2. Setup

You're already in a Git repository (the course repository): we only need to install DVC and initialize it.
`dvc init` creates a `.dvc/` folder containing the DVC configuration and stages it in Git.

In [ ]:
%pip install -q dvc

In [ ]:
# Uncomment if Git doesn't know you yet
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"

In [ ]:
!dvc init
!git status --short

In [ ]:
!git commit -m "Initialize DVC"

## 3. Track a dataset

Let's create a synthetic dataset. It is small to keep this notebook fast, but real datasets often weigh several GB.

In [ ]:
import os
import numpy as np
import pandas as pd

n_rows = 2_000
np.random.seed(42)  # For reproducibility

df = pd.DataFrame({
    'feature1': np.random.randint(0, 1000, size=n_rows),
    'feature2': np.random.randn(n_rows),
    'feature3': np.random.choice(['A', 'B', 'C', 'D'], size=n_rows),
    'feature4': np.random.uniform(0, 1, size=n_rows),
    'feature5': np.random.exponential(1.0, size=n_rows)
})
df.to_csv('sample_dataset.csv', index=False)

size_mb = os.path.getsize('sample_dataset.csv') / (1024 * 1024)
print(f"File created: sample_dataset.csv ({size_mb:.1f} MB)")
print(f"Shape: {df.shape} - columns: {df.columns.tolist()}")

`dvc add` does three things:

1. it copies the file into the DVC cache (`.dvc/cache`), named after its hash;
2. it creates the metafile `sample_dataset.csv.dvc`;
3. it adds the data file to `.gitignore`, so that Git never commits it.

In [ ]:
!dvc add sample_dataset.csv

In [ ]:
!cat sample_dataset.csv.dvc

The `.dvc` file is a tiny text file: its `md5` is the fingerprint of this exact version of the data, and this is what Git versions.
Let's commit it, together with the updated `.gitignore`: this is **version 1** of our dataset.

In [ ]:
!git add sample_dataset.csv.dvc .gitignore
!git commit -m "Track dataset v1 with DVC"

## 4. Modify the dataset and detect changes

We rename a column, add a computed column and filter some rows.

In [ ]:
df = df.rename(columns={'feature3': 'category'})
df['feature6'] = df['feature1'] * df['feature4']   # New computed column
df = df[df['feature1'] > 200]                      # Keep rows where feature1 > 200
df.to_csv('sample_dataset.csv', index=False)
print(f"Shape: {df.shape} - columns: {df.columns.tolist()}")

DVC offers two commands to see what changed:

- `dvc status` compares the data in your workspace with the `.dvc` files: it tells you that the dataset no longer matches its metafile.
- `dvc diff` compares the workspace with a Git commit (`HEAD` by default): it tells you what changed since the last committed version.

In [ ]:
!dvc status

In [ ]:
!dvc diff

Let's record **version 2**. This time, we first enable `core.autostage`: DVC will automatically stage in Git the files it modifies (`.dvc` files, `.gitignore`), so we no longer need `git add`.

In [ ]:
!dvc config core.autostage true
!git add .dvc/config
!dvc add sample_dataset.csv
!git status --short

In [ ]:
!git commit -m "Track dataset v2: renamed feature3, added feature6, filtered rows"

## 5. Go back to a previous version

This is where versioning pays off. Imagine that the v2 dataset degrades your model and you want to retrieve v1. It takes two steps:

1. Git restores the **metafile** of v1: `git checkout <commit> -- sample_dataset.csv.dvc`
2. DVC restores the **data** matching this metafile from its cache: `dvc checkout`

In [ ]:
!git log --oneline -3

In [ ]:
!git checkout HEAD~1 -- sample_dataset.csv.dvc
!dvc checkout sample_dataset.csv.dvc

df = pd.read_csv('sample_dataset.csv')
print(f"Shape: {df.shape} - columns: {df.columns.tolist()}")

We're back to v1: 2,000 rows and the original `feature3` column. Let's return to the latest version the same way:

In [ ]:
!git checkout HEAD -- sample_dataset.csv.dvc
!dvc checkout sample_dataset.csv.dvc

df = pd.read_csv('sample_dataset.csv')
print(f"Shape: {df.shape} - columns: {df.columns.tolist()}")

> The same mechanism works for a whole project: `git checkout <commit>` followed by `dvc checkout` restores both the code **and** the data of any past experiment.

## 6. Track a model

DVC doesn't care about the file format: models, images or archives are tracked exactly like a CSV.
Let's create a (fake) model file and track it. Since autostage is enabled, its `.dvc` file is staged automatically.

In [ ]:
with open('model.pkl', 'wb') as f:
    f.write(b'Model binary content')
print('model.pkl created')

In [ ]:
!dvc add model.pkl
!git status --short

In [ ]:
!git commit -m "Track model with DVC"

## Summary

| Command | What it does |
|---|---|
| `dvc init` | Initializes DVC in a Git repository |
| `dvc add <file>` | Caches the file, creates `<file>.dvc` and adds the file to `.gitignore` |
| `dvc status` | Compares the workspace with the `.dvc` files |
| `dvc diff` | Compares the workspace with a Git commit |
| `git checkout <rev> -- <file>.dvc` + `dvc checkout` | Restores a previous version of the data |
| `dvc config core.autostage true` | Stages DVC files in Git automatically |

For now, the data only lives in your local DVC cache: if you lose your machine, you lose your data. The optional section below pushes it to a remote storage.

# 7. (Optional) Push the data to DagsHub

Just as Git is used with GitHub, DVC needs a remote storage to share data: S3, Google Drive... or DagsHub, which provides a DVC remote for each repository.
The code then goes to GitHub, and the data to DagsHub.

### Prerequisites

1. Your repository must be **on your own GitHub account**. If you cloned the course repository, fork it with the GitHub CLI: `gh repo fork --remote`
   (your fork becomes `origin`, and the course repository `upstream`).
2. Go to [DagsHub](https://dagshub.com/user/sign_up) and sign up with your GitHub account.
3. Click on *Create → New repository → Connect a repository → GitHub*, and pick your repository.

## Configure the DVC remote

The DagsHub client gets a **token** (OAuth login: follow the printed link), which is used as the DVC password: you never write your password in the notebook.
Credentials are written with `--local`, i.e. in `.dvc/config.local`, which is ignored by Git: the token is never committed.

In [ ]:
%pip install -q dagshub

In [ ]:
from dagshub.auth import get_token
from dagshub.common.api import UserAPI

token = get_token()                                  # OAuth login on first call, cached afterwards
dagshub_user = UserAPI.get_current_user().username
repo_name = !basename -s .git $(git remote get-url origin)
repo_name = repo_name[0]

!dvc remote add --default --force origin https://dagshub.com/{dagshub_user}/{repo_name}.dvc
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {dagshub_user}
!dvc remote modify origin --local password {token}
!git add .dvc/config

## Push

The `.dvc` files go to GitHub with `git push`, and the data goes to DagsHub with `dvc push`.

In [ ]:
!git commit -m "Configure DagsHub DVC remote"
!git push -u origin HEAD
!dvc push

Your dataset and model now appear in your DagsHub repository, next to the code. Anyone who clones the repository can fetch them with `dvc pull`.

**Congratulations, you've completed the DVC quickstart!**